# Vintage corpus cleaning: `think-dataset` -> `think-dataset-clean`

This notebook refines [`jbduran/think-dataset`](https://huggingface.co/datasets/jbduran/think-dataset) and writes cleaned shards to [`jbduran/think-dataset-clean`](https://huggingface.co/datasets/jbduran/think-dataset-clean).

It follows the practical filters described in Michael Hla's *Machina Mirabilis* write-up and `gpt1900` scripts:

- Blog: https://michaelhla.com/blog/machina-mirabilis.html
- Scripts: https://github.com/michaelhla/gpt1900/tree/master/scripts/pre1900_scripts

Your settled design choices are baked in:

- Keep whole books as single rows.
- Skip Hla's strict post-1900 physics keyword filter, because this corpus intentionally keeps material through the 1930s.
- Use a moderate GPT-2 token log-prior filter: p2.5-p97.5, targeting roughly 5% prior-filter removal.
- Preserve shard shape: each source `shard_XXXXX.parquet` becomes the same destination shard name.

## Pipeline summary

| Stage | Technique | Expected removal | Time estimate | Resumable? |
|---|---:|---:|---:|---:|
| 0 | Guarded clean command: delete generated destination shards/stats | n/a | <1 min | n/a |
| A | Boilerplate and OCR cleanup | mostly shrinks text; rare drops | included in Stage 2 | yes |
| B | Structural filters: length, printable ratio, OCR artifacts | <1-2% | included in Stage 2 | yes |
| C | GPT-2 token log-prior filter | ~5% | included in Stage 2 | yes |
| 1 | Build or load cached prior table and thresholds | n/a | 10-20 min once | yes |
| 2 | Main shard-by-shard cleaning loop | net ~6-10% docs | ~3-5 h CPU Colab | yes |
| 3 | Aggregate report and provenance upload | n/a | 2-5 min | yes |

The notebook records per-shard document counts, character counts, and removals by reason, then uploads an aggregate `cleaning_report.json` to the output dataset.


## Setup and configuration

Installs the Colab dependencies, authenticates to Hugging Face, and defines all thresholds.

Use a Hugging Face write token with access to the `jbduran` namespace. In Colab, the cleanest path is to add it as a secret named `HF_TOKEN`.


In [ ]:
%pip -q install -U datasets huggingface_hub pyarrow transformers tqdm numpy

import json
import math
import os
import random
import re
import shutil
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import (
    CommitOperationAdd,
    CommitOperationDelete,
    HfApi,
    hf_hub_download,
    login,
)
from tqdm.auto import tqdm
from transformers import GPT2TokenizerFast

try:
    from google.colab import userdata
except Exception:
    userdata = None

print("Dependencies imported.")

In [ ]:
# Source and destination dataset repositories.
SRC_REPO = "jbduran/think-dataset"
DST_REPO = "jbduran/think-dataset-clean"

# Hugging Face auth.
# Recommended in Colab: add a write token as a secret named HF_TOKEN.
HF_TOKEN = None
if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = True

api = HfApi(token=HF_TOKEN)

# Start-over controls. Leave these False for normal resumable runs.
CONFIRM_WIPE = False
FORCE_RECOMPUTE_PRIOR = False

# Structural filters.
MIN_CHARS_RAW = 500
MIN_CHARS_CLEAN = 500
MIN_PRINTABLE = 0.85
MAX_OCR_ARTIFACTS = 50

# Prior filter. p2.5-p97.5 removes about 5% by construction on the sampled docs.
PRIOR_BAND = (2.5, 97.5)
SAMPLE_SHARDS = 12
SAMPLE_DOCS = 20_000
TOKENIZE_CHARS = 50_000
RANDOM_SEED = 42

# Output settings. One input shard maps to one output shard with the same basename.
ROW_GROUP_SIZE = 64
COMPRESSION = "zstd"
COMPRESSION_LEVEL = 3

# Local Colab working directories.
WORK_DIR = Path("/content/think_clean_work")
SRC_CACHE = WORK_DIR / "source"
OUT_DIR = WORK_DIR / "out"
PRIOR_DIR = WORK_DIR / "prior"
for d in [SRC_CACHE, OUT_DIR, PRIOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Source:      {SRC_REPO}")
print(f"Destination: {DST_REPO}")
print(f"Prior band:  p{PRIOR_BAND[0]}-p{PRIOR_BAND[1]}")

## Stage 0: Clean command

This is the requested start-from-scratch command. It deletes generated `shard_*.parquet` files and matching `stats/shard_*.json` files from the destination repo.

It is guarded by `CONFIRM_WIPE = False` in the config cell. Leave it false for normal resume behavior. Set it to true only when you want to rebuild the destination dataset from scratch.


In [ ]:
def list_repo_files_safe(repo_id):
    try:
        return api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    except Exception as exc:
        print(f"Could not list {repo_id}: {exc}")
        return []


def wipe_destination_shards(confirm=False, wipe_prior=False):
    """Delete generated shard/stat/report files from the destination dataset repo."""
    if not confirm:
        print("CONFIRM_WIPE is False; leaving destination repo untouched.")
        print("Set CONFIRM_WIPE = True and re-run this cell to delete generated shards.")
        return

    files = list_repo_files_safe(DST_REPO)
    delete_paths = []
    for path in files:
        is_shard = re.match(r"(^|.*/)shard_\d+\.parquet$", path) is not None
        is_stat = re.match(r"(^|.*/)stats/shard_\d+\.json$", path) is not None
        is_report = path == "cleaning_report.json"
        is_prior = path.startswith("_prior/") if wipe_prior else False
        if is_shard or is_stat or is_report or is_prior:
            delete_paths.append(path)

    if not delete_paths:
        print("No generated shard/stat/report files found to delete.")
        return

    print(f"Deleting {len(delete_paths):,} generated files from {DST_REPO}...")
    for start in range(0, len(delete_paths), 100):
        batch = delete_paths[start:start + 100]
        api.create_commit(
            repo_id=DST_REPO,
            repo_type="dataset",
            operations=[CommitOperationDelete(path_in_repo=p) for p in batch],
            commit_message=f"wipe generated cleaning outputs {start // 100 + 1}",
        )
        print(f"  deleted {start + len(batch):,}/{len(delete_paths):,}")


wipe_destination_shards(confirm=CONFIRM_WIPE, wipe_prior=FORCE_RECOMPUTE_PRIOR)

## Filter library: Stages A and B

This cell defines the text cleanup and structural filters. It is a compact Colab port of the useful Hla `hf_clean.py` ideas for your one-column book dataset.

### Stage A: boilerplate and OCR cleanup

What it does:

- Removes Google Books, HathiTrust, Project Gutenberg, library-stamp, barcode, call-number, and institutional seal fragments.
- Normalizes unicode and common historical/OCR ligatures.
- Rejoins line-break hyphenation and reflows hard-wrapped lines into paragraphs.
- Removes short front-matter stamp lines that often survive after an OCR title/byline.

Pros: removes high-frequency low-information text that small models tend to memorize. It also makes the prior filter less likely to learn boilerplate as "normal."

Cons: regex cleanup is heuristic. It can miss novel OCR corruption and can rarely trim an unusual title page too aggressively.

Expected effect: usually 1-5% fewer characters, with documents dropped only when cleanup leaves too little text.

### Stage B: structural filters

What it does:

- Drops very short raw documents.
- Drops documents with low printable-character ratio.
- Drops documents with many OCR artifact patterns.
- Drops documents that become too short after cleanup.

Pros: cheap, explainable, and catches broken records before expensive tokenization.

Cons: fixed thresholds are blunt and do not understand genre.

Expected removal: typically <1-2%, since the source has already had an initial rough filter.


In [ ]:
OCR_GARBAGE_PATTERNS = [
    r"[■□▪▫●○◆◇★☆♦♣♠♥]+",
    r"[\^~`]{2,}",
    r"[|!]{3,}",
    r"\.{5,}",
    r"\*{5,}",
    r"_{5,}",
    r"-{5,}",
    r"[^\x00-\x7F]{5,}",
    r"[A-Z\s]{50,}",
]

GOOGLE_TEXT_BLOCKS = [
    r"Google's\s+mission\s+is\s+to\s+organize\s+the\s+world's\s+information.*?(?:Google\s+Book\s+Search|search\s+engine).*?\n",
    r"This\s+is\s+a\s+digital\s+copy\s+of\s+a\s+book\s+that\s+was\s+preserved.*?(?:books\.google\.com|public\s+domain|Google\s+Book\s+Search).*?\n",
    r"About\s+Google\s+Book\s+Search.*?(?:books\.google\.com|Book\s+Search).*?\n",
    r"(?:Whether\s+a\s+book\s+is\s+in\s+the\s+public\s+domain|public\s+domain\s+in\s+the\s+United\s+States).*?Google.*?\n",
    r"(?:We\s+encourage\s+the\s+use\s+of\s+public\s+domain|Usage\s+guidelines|Please\s+do\s+not\s+remove\s+this).*?Google.*?\n",
]
GOOGLE_TEXT_COMPILED = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in GOOGLE_TEXT_BLOCKS]

BOILERPLATE_KEYWORDS = [
    "copyright term has expired",
    "merely their custodians",
    "we also ask that you",
    "non-commercial purposes",
    "keep it legal",
    "copyright infringement",
    "discover the world",
    "search through the full text",
    "placing technical restrictions",
    "automated querying",
    "helping authors and publishers",
    "reach new audiences",
    "whether a book is still in copyright",
    "varies from country to country",
    "specific use of any specific book",
    "anywhere in the world",
    "public domain material",
    "hosted by",
    "authorized facsimile",
    "university microfilms",
    "microfilm-xerography",
    "acid-free paper",
    "ann arbor",
    "qooqle",
    "qoogle",
    "optical character recognition",
    "large amount of text",
    "please contact us",
    "in the custody of the",
    "university of california",
    "los angeles",
    "digitized by",
    "digital copy of a book",
    "public domain in the united states",
    "books.google",
    "hathitrust digital library",
    "generated by hathitrust",
]

HATHI_PATTERNS = [
    r"HathiTrust\s+Digital\s+Library[^\n]*\n",
    r"Generated\s+.*?HathiTrust[^\n]*\n",
    r"Public\s+Domain\s+.*?Google-digitized[^\n]*\n",
]

LIBRARY_PATTERNS = [
    r"(?i)university\s+of\s+\w+\s*[•·]\s*\w+",
    r"(?i)the\s+library\s+of\s+the\s+university",
    r"(?i)(?:bequest|gift)\s+of\s+[A-Z][^\n]{0,50}\n",
    r"(?i)public\s+domain.*?google",
    r"(?i)scanned\s+by\s+[^\n]{0,50}\n",
    r"(?i)from\s+the\s+collections?\s+of[^\n]*\n",
]

FRONT_STAMP_PATTERNS = [
    r"(?i)^university\s+of",
    r"(?i)^the\s+library",
    r"(?i)^public\s+library",
    r"(?i)^boston\s+public",
    r"(?i)^property\s+of",
    r"(?i)^in\s+the\s+custody",
    r"(?i)^accession",
    r"(?i)^call\s+n",
    r"(?i)^from\s+the\s+collection",
    r"(?i)^gift\s+of",
    r"(?i)^bequest\s+of",
    r"(?i)^presented\s+by",
    r"(?i)^digitized\s+by",
    r"(?i)^scanned\s+by",
    r"(?i)^oxford\s*$",
    r"(?i)^cambridge\s*$",
    r"(?i)^harvard\s*$",
    r"(?i)^yale\s*$",
    r"(?i)^cornell\s*$",
    r"(?i)^harvard\s+college",
    r"(?i)^harvard\s+university",
    r"(?i)^godfrey\s+lowell\s+cabot",
    r"(?i)^science\s+library",
    r"(?i)^widener\s+library",
    r"(?i)^received",
    r"(?i)^ex\s*libris",
    r"(?i)^sigill",
    r"(?i)^veri\s*tas",
    r"(?i)^ecclesia",
    r"(?i)^nov[\s:]+angl",
    r"(?i)^christo",
    r"(?i)^academiae",
    r"^[A-Z]{1,4}\s+\d{1,5}[\s.]",
    r"^\d{3,7}\s*$",
    r"(?i)^library\s+of\s+congress",
]
FRONT_STAMP_COMPILED = [re.compile(p) for p in FRONT_STAMP_PATTERNS]
STAMP_KEYWORD_COMPILED = [
    re.compile(p) for p in [
        r"(?i)\bharvard\b",
        r"(?i)college\s+library",
        r"(?i)university\s+library",
        r"(?i)ve\w{0,2}r?i?\s*[ts]\w?as",
        r"(?i)academiae",
        r"(?i)sigill\b",
        r"(?i)\bbequest\b",
        r"(?i)\bgift\s+of\b",
        r"(?i)\bclass\s+of\s+\d{4}\b",
    ]
]

PG_START = re.compile(r"\*{3}\s*START\s+OF\s+.*?PROJECT\s+GUTENBERG.*?\*{3}", re.IGNORECASE | re.DOTALL)
PG_END = re.compile(r"\*{3}\s*END\s+OF\s+.*?PROJECT\s+GUTENBERG.*?\*{3}", re.IGNORECASE | re.DOTALL)
HYPHEN_LINEBREAK_RE = re.compile(r"([a-z]{2,})-\n([a-z])")


def estimate_printable_ratio(text, sample_size=10000):
    sample = text[:sample_size]
    if not sample:
        return 0.0
    printable = sum(1 for c in sample if c.isprintable() or c in "\n\t")
    return printable / len(sample)


def count_ocr_artifacts(text):
    sample = text[:30000]
    return sum(len(re.findall(pattern, sample)) for pattern in OCR_GARBAGE_PATTERNS)


def remove_google_boilerplate(text):
    for pattern in GOOGLE_TEXT_COMPILED:
        text = pattern.sub("", text)
    kept = []
    for line in text.split("\n"):
        line_lower = line.lower()
        if "google" in line_lower and (len(line) < 100 or line_lower.find("google") < 50 or line_lower.find("google") > len(line) - 50):
            continue
        if any(kw in line_lower for kw in BOILERPLATE_KEYWORDS):
            continue
        if re.match(r"^\s*[Gg]\s*[Oo0]\s*[Oo0]\s*[Gg]\s*[Ll1]\s*[Ee3]\s*$", line):
            continue
        if line.strip() == "LIBRARY":
            continue
        kept.append(line)
    return "\n".join(kept)


def remove_hathi_boilerplate(text):
    for pattern in HATHI_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return text


def remove_library_stamps(text):
    for pattern in LIBRARY_PATTERNS:
        text = re.sub(pattern, "", text)
    return text


def strip_pg_boilerplate(text):
    start = PG_START.search(text)
    if start:
        text = text[start.end():]
    end = PG_END.search(text)
    if end:
        text = text[:end.start()]
    return text


def normalize_unicode(text):
    text = unicodedata.normalize("NFKC", text)
    replacements = {
        "ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl",
        "æ": "ae", "œ": "oe", "ſ": "s",
        "—": "--", "–": "-",
        "“": '"', "”": '"', "‘": "'", "’": "'",
        "…": "...", "\u00ad": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def normalize_whitespace(text):
    text = text.replace("\t", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_ocr_artifacts(text):
    text = re.sub(r"^\s*[-\[]*\s*\d{1,4}\s*[-\]]*\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[a-zA-Z]\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^[^a-zA-Z]*$", "", text, flags=re.MULTILINE)
    return text


def reflow_text(text):
    paragraphs = text.split("\n\n")
    reflowed = []
    for para in paragraphs:
        line = para.replace("\n", " ")
        line = re.sub(r"  +", " ", line)
        reflowed.append(line.strip())
    return "\n\n".join(p for p in reflowed if p)


def strip_leading_garbage(text, max_lines=100):
    lines = text.split("\n")
    start_idx = 0
    for i, line in enumerate(lines[:max_lines]):
        stripped = line.strip()
        if not stripped:
            continue
        is_stamp = any(p.match(stripped) for p in FRONT_STAMP_COMPILED)
        if not is_stamp and len(stripped) < 80:
            is_stamp = any(p.search(stripped) for p in STAMP_KEYWORD_COMPILED)
        if is_stamp:
            continue
        alpha_count = sum(1 for c in stripped if c.isalpha())
        alpha_ratio = alpha_count / max(len(stripped), 1)
        word_count = len(stripped.split())
        is_content = (
            (len(stripped) > 40 and alpha_ratio > 0.5)
            or (alpha_ratio > 0.8 and word_count >= 3 and len(stripped) > 15)
            or (word_count >= 4 and alpha_ratio > 0.6)
        )
        if is_content:
            start_idx = i
            break
    return "\n".join(lines[start_idx:]) if start_idx > 0 else text


def scrub_front_matter_stamp_lines(text, max_lines=120):
    """Remove short institutional stamp lines that survive after a title/byline."""
    lines = text.split("\n")
    out = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if i < max_lines and len(stripped) < 90:
            if any(p.search(stripped) for p in STAMP_KEYWORD_COMPILED):
                continue
            if re.match(r"^\d[\d\s-]{8,}$", stripped):
                continue
        out.append(line)
    return "\n".join(out)


def remove_front_matter(text, max_scan=15000):
    text = strip_leading_garbage(text)
    text = scrub_front_matter_stamp_lines(text)
    markers = [
        r"(?i)^CHAPTER\s+[IVX\d]+",
        r"(?i)^BOOK\s+[IVX\d]+",
        r"(?i)^PART\s+[IVX\d]+",
        r"(?i)^INTRODUCTION\s*$",
        r"(?i)^PREFACE\s*\.?\s*$",
        r"(?i)^CONTENTS\s*$",
        r"(?i)^THE\s+PREFACE",
        r"(?i)^TO\s+THE\s+READER",
        r"(?i)^ADVERTISEMENT\s*$",
        r"(?i)^DEDICATION\s*$",
        r"(?i)^PROLOGUE\s*$",
    ]
    front = text[:max_scan]
    earliest = max_scan
    for pattern in markers:
        match = re.search(pattern, front, re.MULTILINE)
        if match and match.start() < earliest:
            earliest = match.start()
    return text[earliest:] if 200 < earliest < max_scan else text


def clean_text(text):
    text = remove_google_boilerplate(text)
    text = remove_hathi_boilerplate(text)
    text = remove_library_stamps(text)
    text = strip_pg_boilerplate(text)
    text = normalize_unicode(text)
    text = clean_ocr_artifacts(text)
    text = normalize_whitespace(text)
    text = HYPHEN_LINEBREAK_RE.sub(r"\1\2", text)
    text = reflow_text(text)
    text = remove_front_matter(text)
    return text.strip()


def structural_clean(text):
    if not isinstance(text, str) or len(text) < MIN_CHARS_RAW:
        return None, "too_short_raw"
    if estimate_printable_ratio(text) < MIN_PRINTABLE:
        return None, "low_printable"
    if count_ocr_artifacts(text) > MAX_OCR_ARTIFACTS:
        return None, "ocr_artifacts"
    cleaned = clean_text(text)
    if len(cleaned) < MIN_CHARS_CLEAN:
        return None, "too_short_clean"
    return cleaned, "ok"


def source_shards():
    files = list_repo_files_safe(SRC_REPO)
    shards = sorted([p for p in files if re.match(r"(^|.*/)shard_\d+\.parquet$", p)])
    if not shards:
        shards = sorted([p for p in files if p.endswith(".parquet")])
    print(f"Found {len(shards):,} source parquet shards.")
    return shards


def destination_shards():
    files = list_repo_files_safe(DST_REPO)
    return set(p for p in files if re.match(r"(^|.*/)shard_\d+\.parquet$", p))


def download_source_shard(path_in_repo):
    return hf_hub_download(
        repo_id=SRC_REPO,
        repo_type="dataset",
        filename=path_in_repo,
        token=HF_TOKEN,
        local_dir=SRC_CACHE,
    )


def output_path_for_source(path_in_repo):
    return Path(path_in_repo).name


SOURCE_SHARDS = source_shards()

## Stage 1: Build or load GPT-2 log-prior thresholds

This implements Hla's `prior_filter.py` pattern: tokenize a sample with the GPT-2 tokenizer, count corpus token frequencies, convert them to `log2(count / total)` priors, and score each document by mean token log-prior.

Motivation: very low mean prior often flags rare-token OCR garbage or non-English fragments; very high mean prior often flags repetitive boilerplate or catalog-like text. Good prose tends to sit in the middle.

Pros: no model inference, much cheaper than perplexity filtering, and good at catching statistical outliers that regexes miss.

Cons: it is a quality proxy, not a semantic judge. A legitimate unusual text can be clipped if the band is too narrow.

This notebook uses p2.5-p97.5 for a moderate target of about 5% prior-filter removal. The prior table and thresholds are cached under `_prior/` in the destination repo, so reconnects and reruns load them instead of recomputing.


In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")


def try_download_prior():
    try:
        prior_path = hf_hub_download(DST_REPO, "_prior/log_priors.npy", repo_type="dataset", token=HF_TOKEN)
        thresh_path = hf_hub_download(DST_REPO, "_prior/thresholds.json", repo_type="dataset", token=HF_TOKEN)
        log_priors = np.load(prior_path)
        with open(thresh_path, "r", encoding="utf-8") as f:
            thresholds = json.load(f)
        print("Loaded cached prior table and thresholds from destination repo.")
        return log_priors, thresholds
    except Exception:
        return None, None


def doc_mean_log_prior(text, log_priors):
    ids = tokenizer.encode(text[:TOKENIZE_CHARS], add_special_tokens=False)
    if not ids:
        return None
    ids = [i for i in ids if i < len(log_priors)]
    if not ids:
        return None
    return float(log_priors[ids].mean())


def build_prior_from_sample():
    rng = random.Random(RANDOM_SEED)
    sample_shards = SOURCE_SHARDS[:]
    rng.shuffle(sample_shards)
    sample_shards = sample_shards[:min(SAMPLE_SHARDS, len(sample_shards))]

    docs = []
    stage_estimates = Counter()
    raw_seen = 0

    print(f"Sampling up to {SAMPLE_DOCS:,} docs from {len(sample_shards):,} shards for priors...")
    for shard in tqdm(sample_shards, desc="sample shards"):
        local = download_source_shard(shard)
        pf = pq.ParquetFile(local)
        for rg_idx in range(pf.num_row_groups):
            table = pf.read_row_group(rg_idx, columns=["text"])
            for text in table.column("text").to_pylist():
                raw_seen += 1
                cleaned, reason = structural_clean(text)
                if cleaned is None:
                    stage_estimates[reason] += 1
                    continue
                docs.append(cleaned)
                if len(docs) >= SAMPLE_DOCS:
                    break
            if len(docs) >= SAMPLE_DOCS:
                break
        if len(docs) >= SAMPLE_DOCS:
            break

    if not docs:
        raise RuntimeError("No documents survived the structural sample; loosen thresholds or inspect source data.")

    counts = Counter()
    doc_token_ids = []
    total_tokens = 0
    for text in tqdm(docs, desc="tokenizing sample"):
        ids = tokenizer.encode(text[:TOKENIZE_CHARS], add_special_tokens=False)
        if not ids:
            continue
        counts.update(ids)
        doc_token_ids.append(ids)
        total_tokens += len(ids)

    if total_tokens == 0:
        raise RuntimeError("Tokenization sample produced zero tokens.")

    vocab_size = tokenizer.vocab_size
    log_priors = np.full(vocab_size, np.log2(1.0 / total_tokens), dtype=np.float32)
    for tok_id, count in counts.items():
        if tok_id < vocab_size:
            log_priors[tok_id] = np.log2(count / total_tokens)

    means = np.array([float(log_priors[[i for i in ids if i < vocab_size]].mean()) for ids in doc_token_ids])
    lo, hi = np.percentile(means, PRIOR_BAND)
    thresholds = {
        "prior_band": list(PRIOR_BAND),
        "low": float(lo),
        "high": float(hi),
        "sample_raw_docs_seen": int(raw_seen),
        "sample_structural_kept": int(len(docs)),
        "sample_structural_removed": dict(stage_estimates),
        "sample_prior_docs": int(len(means)),
        "estimated_prior_removed_pct": float(100.0 * ((means < lo) | (means > hi)).mean()),
        "mean_log_prior_percentiles": {
            str(p): float(np.percentile(means, p))
            for p in [1, 2.5, 5, 10, 25, 50, 75, 90, 95, 97.5, 99]
        },
    }

    PRIOR_DIR.mkdir(parents=True, exist_ok=True)
    local_prior = PRIOR_DIR / "log_priors.npy"
    local_thresh = PRIOR_DIR / "thresholds.json"
    local_sample = PRIOR_DIR / "sample_stats.json"
    np.save(local_prior, log_priors)
    for path, payload in [(local_thresh, thresholds), (local_sample, thresholds)]:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, sort_keys=True)

    api.create_commit(
        repo_id=DST_REPO,
        repo_type="dataset",
        operations=[
            CommitOperationAdd(path_in_repo="_prior/log_priors.npy", path_or_fileobj=str(local_prior)),
            CommitOperationAdd(path_in_repo="_prior/thresholds.json", path_or_fileobj=str(local_thresh)),
            CommitOperationAdd(path_in_repo="_prior/sample_stats.json", path_or_fileobj=str(local_sample)),
        ],
        commit_message="add cleaning prior table and thresholds",
    )

    print("Sample-based prior estimates:")
    print(json.dumps(thresholds, indent=2))
    return log_priors, thresholds


if not FORCE_RECOMPUTE_PRIOR:
    LOG_PRIORS, PRIOR_THRESHOLDS = try_download_prior()
else:
    LOG_PRIORS, PRIOR_THRESHOLDS = None, None

if LOG_PRIORS is None:
    LOG_PRIORS, PRIOR_THRESHOLDS = build_prior_from_sample()

PRIOR_LOW = PRIOR_THRESHOLDS["low"]
PRIOR_HIGH = PRIOR_THRESHOLDS["high"]
print(f"Using mean log-prior band [{PRIOR_LOW:.4f}, {PRIOR_HIGH:.4f}]")

## Stage 2: Main resumable shard loop

For every source shard, the notebook downloads the parquet, cleans and filters every whole-book row, writes a same-named output shard, and uploads the shard plus its stats JSON in one commit.

Resume behavior: if `shard_XXXXX.parquet` already exists in the destination repo, the loop skips it. A crash or Colab disconnect costs at most the shard currently in flight.

Each shard stats file records:

- input documents
- kept documents
- removals by reason
- raw characters
- cleaned characters before prior filtering
- kept characters

Expected result: same shard count and names as the original dataset, with roughly 90-94% of documents/characters retained depending on the actual source distribution.


In [ ]:
def process_one_text(text):
    cleaned, reason = structural_clean(text)
    if cleaned is None:
        return None, reason, None, len(text) if isinstance(text, str) else 0, 0

    mean_prior = doc_mean_log_prior(cleaned, LOG_PRIORS)
    if mean_prior is None:
        return None, "empty_tokenization", None, len(text), len(cleaned)
    if mean_prior < PRIOR_LOW:
        return None, "prior_low", mean_prior, len(text), len(cleaned)
    if mean_prior > PRIOR_HIGH:
        return None, "prior_high", mean_prior, len(text), len(cleaned)
    return cleaned, "ok", mean_prior, len(text), len(cleaned)


def write_parquet_texts(texts, local_out):
    table = pa.table({"text": texts})
    pq.write_table(
        table,
        local_out,
        row_group_size=ROW_GROUP_SIZE,
        compression=COMPRESSION,
        compression_level=COMPRESSION_LEVEL,
    )


def process_shard(path_in_repo):
    output_name = output_path_for_source(path_in_repo)
    local_source = download_source_shard(path_in_repo)
    local_out = OUT_DIR / output_name
    local_stats = OUT_DIR / f"{Path(output_name).stem}.json"

    stats = {
        "source_shard": path_in_repo,
        "output_shard": output_name,
        "started_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_input": 0,
        "n_kept": 0,
        "n_removed": 0,
        "removed_by_reason": defaultdict(int),
        "chars_raw": 0,
        "chars_after_clean_before_prior": 0,
        "chars_kept": 0,
        "mean_prior_kept_sum": 0.0,
    }

    kept_texts = []
    pf = pq.ParquetFile(local_source)
    for rg_idx in range(pf.num_row_groups):
        table = pf.read_row_group(rg_idx, columns=["text"])
        for text in table.column("text").to_pylist():
            stats["n_input"] += 1
            cleaned, reason, mean_prior, raw_chars, clean_chars = process_one_text(text)
            stats["chars_raw"] += raw_chars
            stats["chars_after_clean_before_prior"] += clean_chars
            if cleaned is None:
                stats["n_removed"] += 1
                stats["removed_by_reason"][reason] += 1
                continue
            kept_texts.append(cleaned)
            stats["n_kept"] += 1
            stats["chars_kept"] += len(cleaned)
            stats["mean_prior_kept_sum"] += float(mean_prior)

    stats["removed_by_reason"] = dict(stats["removed_by_reason"])
    stats["mean_prior_kept"] = (
        stats["mean_prior_kept_sum"] / stats["n_kept"] if stats["n_kept"] else None
    )
    del stats["mean_prior_kept_sum"]
    stats["finished_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

    write_parquet_texts(kept_texts, local_out)
    with open(local_stats, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, sort_keys=True)

    api.create_commit(
        repo_id=DST_REPO,
        repo_type="dataset",
        operations=[
            CommitOperationAdd(path_in_repo=output_name, path_or_fileobj=str(local_out)),
            CommitOperationAdd(path_in_repo=f"stats/{Path(output_name).stem}.json", path_or_fileobj=str(local_stats)),
        ],
        commit_message=f"clean {output_name}",
    )
    return stats


done = destination_shards()
todo = [s for s in SOURCE_SHARDS if output_path_for_source(s) not in done]
print(f"Already completed: {len(SOURCE_SHARDS) - len(todo):,}/{len(SOURCE_SHARDS):,}")
print(f"Remaining shards:   {len(todo):,}")

all_stats = []
t0 = time.time()
for shard in tqdm(todo, desc="cleaning shards"):
    try:
        st = process_shard(shard)
        all_stats.append(st)
        removed_pct = 100.0 * st["n_removed"] / max(st["n_input"], 1)
        print(
            f"{st['output_shard']}: kept {st['n_kept']:,}/{st['n_input']:,} "
            f"removed={removed_pct:.2f}% reasons={st['removed_by_reason']}"
        )
    except Exception as exc:
        print(f"ERROR while processing {shard}: {exc}")
        raise

elapsed = time.time() - t0
print(f"Stage 2 complete for this run in {elapsed / 3600:.2f} h.")

## Stage 3: Aggregate report and provenance

Reads all uploaded per-shard stats, prints exact removal counts and percentages, writes `cleaning_report.json`, and updates the destination repo README.

This stage is safe to run mid-pipeline. It reports whatever shard set has completed so far.


In [ ]:
def download_json_file(path_in_repo):
    local = hf_hub_download(DST_REPO, path_in_repo, repo_type="dataset", token=HF_TOKEN)
    with open(local, "r", encoding="utf-8") as f:
        return json.load(f)


files = list_repo_files_safe(DST_REPO)
stat_files = sorted([p for p in files if re.match(r"stats/shard_\d+\.json$", p)])
print(f"Found {len(stat_files):,} completed shard stat files.")

totals = {
    "n_shards_done": len(stat_files),
    "n_input": 0,
    "n_kept": 0,
    "n_removed": 0,
    "chars_raw": 0,
    "chars_after_clean_before_prior": 0,
    "chars_kept": 0,
    "removed_by_reason": Counter(),
    "source_repo": SRC_REPO,
    "destination_repo": DST_REPO,
    "prior_thresholds": PRIOR_THRESHOLDS,
    "settings": {
        "min_chars_raw": MIN_CHARS_RAW,
        "min_chars_clean": MIN_CHARS_CLEAN,
        "min_printable": MIN_PRINTABLE,
        "max_ocr_artifacts": MAX_OCR_ARTIFACTS,
        "prior_band": list(PRIOR_BAND),
        "whole_books_kept": True,
        "post_1900_physics_filter": "skipped for 1930s cutoff",
    },
}

for sf in tqdm(stat_files, desc="stats"):
    st = download_json_file(sf)
    for k in ["n_input", "n_kept", "n_removed", "chars_raw", "chars_after_clean_before_prior", "chars_kept"]:
        totals[k] += int(st.get(k, 0))
    totals["removed_by_reason"].update(st.get("removed_by_reason", {}))

totals["removed_by_reason"] = dict(totals["removed_by_reason"])
totals["docs_removed_pct"] = 100.0 * totals["n_removed"] / max(totals["n_input"], 1)
totals["docs_kept_pct"] = 100.0 * totals["n_kept"] / max(totals["n_input"], 1)
totals["chars_kept_pct_vs_raw"] = 100.0 * totals["chars_kept"] / max(totals["chars_raw"], 1)
totals["updated_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

print(json.dumps(totals, indent=2, sort_keys=True))

report_path = OUT_DIR / "cleaning_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(totals, f, indent=2, sort_keys=True)

readme = f"""---
dataset_info:
  features:
  - name: text
    dtype: string
---

# think-dataset-clean

Cleaned version of `jbduran/think-dataset`, produced with a Colab notebook based on Michael Hla's Machina Mirabilis / gpt1900 filtering approach.

Design choices:

- Whole books are preserved as rows.
- The post-1900 physics keyword filter is skipped because this dataset intentionally keeps texts up to the 1930s.
- A moderate GPT-2 token log-prior band is used: p{PRIOR_BAND[0]}-p{PRIOR_BAND[1]}.
- Each source shard maps to one output shard of the same basename.

Current report:

- Shards completed: {totals["n_shards_done"]:,}
- Input documents seen: {totals["n_input"]:,}
- Kept documents: {totals["n_kept"]:,} ({totals["docs_kept_pct"]:.2f}%)
- Removed documents: {totals["n_removed"]:,} ({totals["docs_removed_pct"]:.2f}%)
- Kept characters vs raw: {totals["chars_kept_pct_vs_raw"]:.2f}%

Removal reasons:

{chr(10).join(f"- {reason}: {count:,}" for reason, count in sorted(totals["removed_by_reason"].items(), key=lambda x: -x[1]))}
"""

readme_path = OUT_DIR / "README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

api.create_commit(
    repo_id=DST_REPO,
    repo_type="dataset",
    operations=[
        CommitOperationAdd(path_in_repo="cleaning_report.json", path_or_fileobj=str(report_path)),
        CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=str(readme_path)),
    ],
    commit_message="update cleaning report",
)

print("Uploaded cleaning_report.json and README.md.")

## Notes for reruns

- Disconnected or crashed: rerun the notebook; Stage 1 loads cached priors and Stage 2 skips completed shards.
- Start over: set `CONFIRM_WIPE = True` and run Stage 0. Also set `FORCE_RECOMPUTE_PRIOR = True` if you changed prior sampling or thresholds.
- Too aggressive or too weak: adjust `PRIOR_BAND`. `(1, 99)` removes about 2%; `(5, 95)` removes about 10%.
- Output format: single `text` column parquet shards, same shard names as source, ZSTD compression.
